In [1]:
# build a pytorch lstm model
import numpy as np
import torch
import torch.nn as nn
from brevitas.nn import QuantLinear, QuantLSTM, QuantIdentity
from common import weight_quantizer, act_quantizer, Int8ActPerTensorFloatScratch

w_bit = 2; io_bit = 2; h_bit = 8; acc_bit = 8; a_bit = 8; r_bit = 16


In [2]:

class QLSTM(nn.Module):
    def __init__(self, input_size=8, hidden_size=32, num_layers=1, num_classes=8):
        """
        Build LSTM model
        Args:
            input_size: Input feature dimension (number of EMG sensors)
            hidden_size: LSTM hidden layer size
            num_layers: Number of LSTM layers
            num_classes: Number of classification categories (number of gestures)
        """
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = QuantLSTM(
                    input_size, hidden_size, num_layers, batch_first=True,
                    weight_quant = weight_quantizer['int{}'.format(w_bit)],
                    io_quant=act_quantizer['int{}'.format(io_bit)],
                    hidden_state_output_quant = act_quantizer['int{}'.format(h_bit)],
                    gate_acc_quant = act_quantizer['int{}'.format(acc_bit)],
                    sigmoid_quant = act_quantizer['uint{}'.format(a_bit)],
                    tanh_quant = act_quantizer['int{}'.format(a_bit)],
                    cell_state_quant = act_quantizer['int{}'.format(r_bit)]
                )
        self.quant_identity = QuantIdentity(act_quant=Int8ActPerTensorFloatScratch, 
                                        return_quant_tensor = True)
        
        self.fc = QuantLinear(hidden_size, num_classes, weight_quant=weight_quantizer['int8'])
    
    def forward(self, x):
        """
        forward pass
        Args:
            x: input tensor (batch_size, seq_len, input_size)
        Returns:
            output: model output
        """

        out, (h0, c0) = self.lstm(x)
        
        # take the output of the last time step
        out = out[:, -1, :]
        
        # fully connected layer
        out = self.fc(out)
        return out

In [3]:
from EMG.dataset import EGMDataset
from torch.utils.data import DataLoader

DATA_PATH = "EMG"
val_dataset = EGMDataset(DATA_PATH, train=False)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

X_test, y_test = next(iter(val_loader))
X_test = X_test.float()
y_test = y_test.long()

print(X_test.shape)
print(y_test.shape)

torch.Size([160, 100, 8])
torch.Size([160])


In [4]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """training one epoch"""
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for features, labels in train_loader:
        features = features.float().to(device)  # ensure data type is float
        labels = labels.long().to(device)  # ensure labels are long type

        output = model(features)
        loss = criterion(output, labels)
        
        # backpropagation and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # calculate accuracy
        _, predicted = torch.max(output.data, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device):
    """validate model"""
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for features, labels in val_loader:
            features = features.float().to(device)
            labels = labels.long().to(device)
            
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            _, predicted = torch.max(outputs.data, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            total_loss += loss.item()
    
    avg_loss = total_loss / len(val_loader)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


def main():
    # training parameters
    BATCH_SIZE = 32
    EPOCHS = 50
    LEARNING_RATE = 0.001
    DATA_PATH = "EMG"  # EMG dataset path
    
    # set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # load dataset
    train_dataset = EGMDataset(DATA_PATH, train=True)
    val_dataset = EGMDataset(DATA_PATH, train=False)
    
    # create data loader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # create model
    model = QLSTM(input_size=8).to(device)  # EMG has 8 sensors
    
    # define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # training history
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    best_val_acc = 0
    
    # training loop
    for epoch in range(EPOCHS):
        # training
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # record history
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
        # save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_emg_model.pth')
        
        print(f'Epoch [{epoch+1}/{EPOCHS}]')
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')    
    
    # load best model for testing
    from brevitas import config as brevitas_config
    brevitas_config.IGNORE_MISSING_KEYS=True
    model.load_state_dict(torch.load('best_emg_model.pth'))
    model.to(device)
    test_loss, test_acc = validate(model, val_loader, criterion, device)
    print(f'\nTest accuracy: {test_acc:.4f}')
    print(f'Test loss: {test_loss:.4f}')
    
    # calculate accuracy of each class
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for features, labels in val_loader:
            features = features.float().to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    
    # calculate accuracy of each gesture
    for i in range(8):  # 8 gestures
        mask = (all_labels == i)
        class_acc = np.mean(all_predictions[mask] == all_labels[mask])
        print(f"Gesture {i} accuracy: {class_acc:.4f}")

In [5]:
main()

Using device: cuda


/home/robin/miniconda3/envs/torch/lib/python3.8/site-packages/torch/_tensor.py:1413: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1925.)
  return super().rename(names)


Epoch [1/50]
Train Loss: 2.0941, Train Acc: 0.1000
Val Loss: 2.0659, Val Acc: 0.3000
Epoch [2/50]
Train Loss: 2.0519, Train Acc: 0.3167
Val Loss: 2.0242, Val Acc: 0.4000
Epoch [3/50]
Train Loss: 2.0074, Train Acc: 0.3958
Val Loss: 1.9791, Val Acc: 0.4500
Epoch [4/50]
Train Loss: 1.9601, Train Acc: 0.4375
Val Loss: 1.9240, Val Acc: 0.4750
Epoch [5/50]
Train Loss: 1.9026, Train Acc: 0.4542
Val Loss: 1.8699, Val Acc: 0.4750
Epoch [6/50]
Train Loss: 1.8516, Train Acc: 0.4708
Val Loss: 1.8150, Val Acc: 0.4750
Epoch [7/50]
Train Loss: 1.7953, Train Acc: 0.4833
Val Loss: 1.7568, Val Acc: 0.4875
Epoch [8/50]
Train Loss: 1.7404, Train Acc: 0.4875
Val Loss: 1.6806, Val Acc: 0.4938
Epoch [9/50]
Train Loss: 1.6481, Train Acc: 0.5000
Val Loss: 1.6143, Val Acc: 0.5625
Epoch [10/50]
Train Loss: 1.5754, Train Acc: 0.5417
Val Loss: 1.5321, Val Acc: 0.6000
Epoch [11/50]
Train Loss: 1.5021, Train Acc: 0.5792
Val Loss: 1.4406, Val Acc: 0.6250
Epoch [12/50]
Train Loss: 1.4120, Train Acc: 0.5917
Val Loss: 1

/tmp/ipykernel_336980/3331304085.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_emg_model.pth'))



Test accuracy: 0.9750
Test loss: 0.3487
Gesture 0 accuracy: 0.9000
Gesture 1 accuracy: 0.9500
Gesture 2 accuracy: 1.0000
Gesture 3 accuracy: 0.9500
Gesture 4 accuracy: 1.0000
Gesture 5 accuracy: 1.0000
Gesture 6 accuracy: 1.0000
Gesture 7 accuracy: 1.0000
